# Model Comparison
We will compare the baseline model with other models using ROC AUC and other evaluation metrics. To make the comparison fair, we will merge the transaction and identity datasets, create two custom features, and use the same train–validation split for every model...

* `HasIdentity`: Indicates whether identity information is available for the transaction.
* `TransHour`: Represents the hour of the transaction.

In [ ]:
# Data processing
import pandas as pd
import numpy as np

# ML model
from catboost import CatBoostClassifier

# Loading the data 
train_transaction = pd.read_csv("../../Data/train_transaction.csv",low_memory=False)
train_identity =    pd.read_csv("../../Data/train_identity.csv",low_memory=False)

print("Train transaction shape:", train_transaction.shape)
print("Train identity shape:", train_identity.shape)

#Train transaction shape: (590540, 394)
#Train identity shape: (144233, 41)

train_identity["HasIdentity"] = 1
train_merged = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)
train_merged["HasIdentity"] = train_merged["HasIdentity"].fillna(0).astype("int8")
train_merged["TranHour"] = ((train_merged["TransactionDT"] // 3600) % 24).astype("int8")

print("Merged shape:", train_merged.shape)
print(train_merged["HasIdentity"].value_counts())
print("Hour range:", train_merged["TranHour"].min(), "-", train_merged["TranHour"].max())

In [3]:
# Train-Validation Split

# Confirm that transactions are ordered by time
print(train_merged["TransactionDT"].is_monotonic_increasing)

split_index = int(len(train_merged) * 0.80)

# Select the model features
excluded_columns = ["TransactionID", "isFraud", "TransactionDT"]
baseline_features = [
    column for column in train_merged.columns
    if column not in excluded_columns
]

X_train = train_merged.iloc[:split_index][baseline_features]
y_train = train_merged.iloc[:split_index]["isFraud"]
X_validation = train_merged.iloc[split_index:][baseline_features]
y_validation = train_merged.iloc[split_index:]["isFraud"]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("NO. of features:", len(baseline_features))
print("Train fraud rate:", round(y_train.mean() * 100, 3), "%")
print("Validation fraud rate:", round(y_validation.mean() * 100, 3), "%")


True
X_train: (472432, 433)
X_validation: (118108, 433)
NO. of features: 433
Train fraud rate: 3.514 %
Validation fraud rate: 3.441 %


In [ ]:
# Idintify the Feature/column types either Cat or No .. 
# Feature Types - Numerical or categorical ? 
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
# Any column is not in categorical_features  will be numric 
numeric_features = [
    column for column in X_train.columns
    if column not in categorical_features
]
print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

Categorical features: 31
Numeric features: 402


# Dummy Baseline
The Dummy Classifier ignores the input features and provides a minimum baseline for evaluating the real models.

In [ ]:
# Dummy Baseline


from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(X_train, y_train)

dummy_probability = dummy_model.predict_proba(X_validation)[:, 1]

model_comparison = pd.DataFrame([{
    "Model": "Dummy Baseline",
    "ROC_AUC": roc_auc_score(y_validation, dummy_probability),
    "Average_Precision": average_precision_score(y_validation, dummy_probability)
}])

display(model_comparison.round(3))
# it is random 
# Roc_AUC = 0.5 , AVG_PREcision = 0.0344



,Model,ROC_AUC,Average_Precision
0,Dummy Baseline,0.5,0.034


# Logistic Regression: linear baseline
*Numerical missing values are replaced with the median
*Prserving missingness indicators
*Categorical features are one-hot encoded

In [ ]:
#  Logistic Regression: linear baseline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(
    ColumnTransformer([
        ("numeric",
         make_pipeline(
             # strategy="median": replace the missing valu with Median, 
             # add_indicator=True: the value is missing and replcaed with the median.
             SimpleImputer(strategy="median", add_indicator=True),

             StandardScaler()
         ),

         # we will consider the TranHour as cat not number .. 
         [column for column in numeric_features if column != "TranHour"]),

        ("categorical",
         make_pipeline(
             # For missing cat data, we will give it constant value (missing)
             SimpleImputer(strategy="constant", fill_value="Missing"),
             # Convert the cat data to 0 and 1 columns .. 
             OneHotEncoder(handle_unknown="ignore")
         ),
         categorical_features + ["TranHour"])
    ]),

    LogisticRegression(
        solver="saga",
        class_weight="balanced",
        max_iter=200,
        random_state=42
    )
)

logistic_model.fit(X_train, y_train)

logistic_probability = logistic_model.predict_proba(X_validation)[:, 1]

model_comparison.loc[len(model_comparison)] = [
    "Logistic Regression",
    roc_auc_score(y_validation, logistic_probability),
    average_precision_score(y_validation, logistic_probability)
]

display(model_comparison.round(4))

In [ ]:
# Logistic Model -- CONT..


logistic_model.set_params(
    # The model reached max_iter without converging.. so we used (warm_start=True) to continue from the previous coefficient
    logisticregression__warm_start=True,
    logisticregression__max_iter=500
)

logistic_model.fit(X_train, y_train)
logistic_probability = logistic_model.predict_proba(X_validation)[:, 1]

# Update the existing Logistic Regression row
model_comparison.loc[model_comparison["Model"] == "Logistic Regression",["ROC_AUC", "Average_Precision"]
] = [
    roc_auc_score(y_validation, logistic_probability),
    average_precision_score(y_validation, logistic_probability)
]

display(model_comparison.round(3))

# Logistic Regression Result
Logistic Regression reached the maximum of 500 iterations before full mathematical convergence
However, the validation results changed only slightly between 200 and 500 iterations

 no additional iterations were performed ..

- ROC-AUC: 0.834
- Average Precision: 0.184

In [ ]:
# LightGBM Model

#%pip install lightgbm
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
X_train_lgbm = X_train.copy(deep=False)
X_validation_lgbm = X_validation.copy(deep=False)

for column in categorical_features + ["TranHour"]:
    X_train_lgbm[column] = X_train_lgbm[column].astype("category")

    X_validation_lgbm[column] = pd.Categorical(
        X_validation_lgbm[column],
        categories=X_train_lgbm[column].cat.categories
    )
 

In [ ]:
# LightGBM Model

lgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_lgbm,
    y_train,
    eval_set=[(X_validation_lgbm, y_validation)],
    eval_metric="auc",
    callbacks=[
        early_stopping(100),
        log_evaluation(100)
    ]
)

In [ ]:
# LightGBM - storing the result in our comparison table 

lgbm_probability = lgbm_model.predict_proba(X_validation_lgbm)[:, 1]

model_comparison.loc[len(model_comparison)] = [
    "LightGBM",
    roc_auc_score(y_validation, lgbm_probability),
    average_precision_score(y_validation, lgbm_probability)
]

print("Best iteration:", lgbm_model.best_iteration_)
display(model_comparison.round(4))

# Keep the latest result for each model
#model_comparison = model_comparison.drop_duplicates(subset=["Model"], keep="last").reset_index(drop=True)
#display(model_comparison.round(4))



Best iteration: 715


,Model,ROC_AUC,Average_Precision
0,Dummy Baseline,0.5000,0.0344
1,Logistic Regression,0.8338,0.1840
2,CatBoost,0.9007,0.5562
3,LightGBM,0.9117,0.5615
4,LightGBM,0.9117,0.5615


# Extended CatBoost Training

The previous CatBoost model reached its best result near the maximum iteration limit
To provide a fair comparison with LightGBM, CatBoost is trained again with a higher iteration from 600 to 2000 limit and the same early-stopping value 
All other CatBoost parameters remain unchanged

In [ ]:
# catboost - Trainin in iterations= 2000
X_train_catboost = X_train.copy(deep=False)
X_validation_catboost = X_validation.copy(deep=False)

for column in categorical_features:
    X_train_catboost[column] = ( X_train_catboost[column].fillna("Missing").astype(str))
    X_validation_catboost[column] = (X_validation_catboost[column].fillna("Missing").astype(str) )

catboost_model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100
)

catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_validation_catboost, y_validation),
    use_best_model=True,
    early_stopping_rounds=100
)

0:	test: 0.7111116	best: 0.7111116 (0)	total: 716ms	remaining: 23m 50s
100:	test: 0.8744384	best: 0.8744685 (99)	total: 57.4s	remaining: 17m 59s
200:	test: 0.8845463	best: 0.8849843 (199)	total: 2m 5s	remaining: 18m 40s
300:	test: 0.8868078	best: 0.8868525 (299)	total: 3m 14s	remaining: 18m 18s
400:	test: 0.8902620	best: 0.8906656 (382)	total: 4m 25s	remaining: 17m 36s
500:	test: 0.8919148	best: 0.8919691 (497)	total: 5m 37s	remaining: 16m 50s
600:	test: 0.8939685	best: 0.8941068 (589)	total: 6m 50s	remaining: 15m 54s
700:	test: 0.8961484	best: 0.8961674 (688)	total: 8m 2s	remaining: 14m 53s
800:	test: 0.8972078	best: 0.8972516 (798)	total: 9m 14s	remaining: 13m 50s
900:	test: 0.8976080	best: 0.8976800 (831)	total: 10m 26s	remaining: 12m 44s
1000:	test: 0.8988441	best: 0.8988660 (986)	total: 11m 43s	remaining: 11m 42s
1100:	test: 0.8995873	best: 0.8996016 (1098)	total: 12m 52s	remaining: 10m 30s
1200:	test: 0.9003725	best: 0.9004218 (1183)	total: 14m 1s	remaining: 9m 19s
1300:	test: 0.

CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=2000, learning_rate=0.1, loss_function='Logloss', random_seed=42, task_type='CPU', verbose=100)

In [ ]:
# catboost - storing the result in our comparison table 
catboost_probability = catboost_model.predict_proba(X_validation_catboost)[:, 1]
model_comparison.loc[
    model_comparison["Model"] == "CatBoost",
    ["ROC_AUC", "Average_Precision"]
] = [
    roc_auc_score(y_validation, catboost_probability),
    average_precision_score(y_validation, catboost_probability)
]
print("CatBoost best iteration:", catboost_model.get_best_iteration())
display(model_comparison.sort_values("Average_Precision", ascending=False).round(4))

CatBoost best iteration: 1227


,Model,ROC_AUC,Average_Precision
3,LightGBM,0.9117,0.5615
2,CatBoost,0.9007,0.5562
1,Logistic Regression,0.8338,0.1840
0,Dummy Baseline,0.5000,0.0344


# Model Selection
[LightGBM] was selected as the best-performing model on the (chronological) validation set
* achieved the highest [ROC-AUC] and ِAP [AveragePrecision]
* has less training time than the other candidate models

# Final Model Comparison

Model Name              |   ROC-AUC    |   Average Precision

LightGBM                |   0.9117     |     0.5615

CatBoost                |   0.9007     |     0.5562

Logistic Regression     |   0.8338     |     0.1840

Dummy Baseline          |   0.5000     |     0.0344



CatBoost was given a higher iteration limit with early stopping before the final comparison
Its best iteration was [1227]

In [ ]:
# LightGBM - Evaluation Metrics 

from sklearn.metrics import (precision_score,recall_score,f1_score,confusion_matrix)

threshold_rows = []

for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:

    prediction = (lgbm_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        prediction,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Predicted_Fraud": prediction.sum(),
        "True_Positive": tp,
        "False_Positive": fp,
        "False_Negative": fn,
        "True_Negative": tn,
        "Precision": precision_score(y_validation, prediction, zero_division=0),
        "Recall": recall_score(y_validation, prediction, zero_division=0),
        "F1_Score": f1_score(y_validation, prediction, zero_division=0),
        "Alert_Rate_Percent": prediction.mean() * 100
    })

threshold_comparison = pd.DataFrame(threshold_rows)

display(threshold_comparison.round(4))

# 0.2 threshold has the best F1 socre [0.5387]

,Threshold,Predicted_Fraud,True_Positive,False_Positive,False_Negative,True_Negative,Precision,Recall,F1_Score,Alert_Rate_Percent
0,0.5,1692,1385,307,2679,113737,0.8186,0.3408,0.4812,1.4326
1,0.4,2011,1536,475,2528,113569,0.7638,0.3780,0.5057,1.7027
2,0.3,2520,1739,781,2325,113263,0.6901,0.4279,0.5283,2.1336
3,0.2,3424,2017,1407,2047,112637,0.5891,0.4963,0.5387,2.8990
4,0.1,5754,2484,3270,1580,110774,0.4317,0.6112,0.5060,4.8718


In [ ]:
# lightgbm -- Feature importance ..

lightgbm_feature_importance = pd.DataFrame({
    "Feature": X_train_lgbm.columns,
    "Importance": lgbm_model.booster_.feature_importance(
        importance_type="gain"
    )
})

# Convert importance into percentages
lightgbm_feature_importance["Importance"] = (
    lightgbm_feature_importance["Importance"]
    / lightgbm_feature_importance["Importance"].sum()
    * 100
)

lightgbm_feature_importance = (
    lightgbm_feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

lightgbm_feature_importance["Cumulative_Importance"] = (
    lightgbm_feature_importance["Importance"].cumsum()
)

display(lightgbm_feature_importance.head(20).round(4))

,Feature,Importance,Cumulative_Importance
0,V258,12.8137,12.8137
1,C1,6.0389,18.8526
2,DeviceInfo,5.4620,24.3147
3,C14,4.8979,29.2126
4,V294,2.9584,32.1709
5,R_emaildomain,2.7998,34.9707
6,C13,2.6866,37.6573
7,card2,2.5740,40.2313
8,D2,2.3381,42.5694
9,TransactionAmt,2.1939,44.7633


## Limited LightGBM Hyperparameter Tuning

A limited randomized search is performed using temporal cross-validation.

- Only the training period is used during tuning.
- Transaction order is preserved.
- ROC-AUC is the primary selection metric.
- Average Precision is monitored as a secondary metric.
- The final validation period remains outside the tuning process.

In [21]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# Temporal cross-validation without shuffling
time_cv = TimeSeriesSplit(n_splits=3)

lgbm_tuning_model = LGBMClassifier(
    objective="binary",
    n_estimators=lgbm_model.best_iteration_,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_parameters = {
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 8, 12],
    "min_child_samples": [20, 50, 100],
    "colsample_bytree": [0.75, 0.90, 1.00],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [0, 1, 5]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_tuning_model,
    param_distributions=lgbm_parameters,
    n_iter=8,
    scoring={
        "ROC_AUC": "roc_auc",
        "Average_Precision": "average_precision"
    },
    refit="ROC_AUC",
    cv=time_cv,
    random_state=42,
    n_jobs=1,
    verbose=2
)

lgbm_search.fit(X_train_lgbm, y_train)

print("Best CV ROC-AUC:", round(lgbm_search.best_score_, 4))
print(
    "Average Precision:",
    round(
        lgbm_search.cv_results_["mean_test_Average_Precision"][
            lgbm_search.best_index_
        ],
        4
    )
)
print("\nBest parameters:")
print(lgbm_search.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.9, max_depth=12, min_child_samples=100, num_leaves=31, reg_alpha=0, reg_lambda=0; total time=  11.2s
[CV] END colsample_bytree=0.9, max_depth=12, min_child_samples=100, num_leaves=31, reg_alpha=0, reg_lambda=0; total time=  15.1s
[CV] END colsample_bytree=0.9, max_depth=12, min_child_samples=100, num_leaves=31, reg_alpha=0, reg_lambda=0; total time=  29.6s
[CV] END colsample_bytree=0.75, max_depth=8, min_child_samples=100, num_leaves=31, reg_alpha=0.1, reg_lambda=1; total time=  17.1s
[CV] END colsample_bytree=0.75, max_depth=8, min_child_samples=100, num_leaves=31, reg_alpha=0.1, reg_lambda=1; total time=  30.8s
[CV] END colsample_bytree=0.75, max_depth=8, min_child_samples=100, num_leaves=31, reg_alpha=0.1, reg_lambda=1; total time=  44.4s
[CV] END colsample_bytree=0.9, max_depth=-1, min_child_samples=100, num_leaves=15, reg_alpha=0.1, reg_lambda=5; total time=  22.8s
[CV] END colsample_bytree=0.9

In [ ]:
# Train LightGBM using the best parameters from temporal cross-validation
lgbm_tuned_model = LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    **lgbm_search.best_params_
)

lgbm_tuned_model.fit(
    X_train_lgbm,
    y_train,
    eval_set=[(X_validation_lgbm, y_validation)],
    eval_metric="auc",
    callbacks=[
        early_stopping(100),
        log_evaluation(100)
    ]
)

In [23]:
# Predict fraud probabilities
lgbm_tuned_probability = lgbm_tuned_model.predict_proba(
    X_validation_lgbm
)[:, 1]

# Compare the current and tuned LightGBM models
lgbm_tuning_comparison = pd.DataFrame({
    "Model": [
        "LightGBM Current",
        "LightGBM Tuned"
    ],
    "Best_Iteration": [
        lgbm_model.best_iteration_,
        lgbm_tuned_model.best_iteration_
    ],
    "ROC_AUC": [
        roc_auc_score(y_validation, lgbm_probability),
        roc_auc_score(y_validation, lgbm_tuned_probability)
    ],
    "Average_Precision": [
        average_precision_score(y_validation, lgbm_probability),
        average_precision_score(y_validation, lgbm_tuned_probability)
    ]
})

display(lgbm_tuning_comparison.round(4))

,Model,Best_Iteration,ROC_AUC,Average_Precision
0,LightGBM Current,715,0.9117,0.5615
1,LightGBM Tuned,379,0.9181,0.5717


In [24]:
# Remove the previous tuned result before adding the updated one
model_comparison = model_comparison[
    model_comparison["Model"] != "LightGBM Tuned"
].copy()

model_comparison.loc[len(model_comparison)] = [
    "LightGBM Tuned",
    roc_auc_score(y_validation, lgbm_tuned_probability),
    average_precision_score(y_validation, lgbm_tuned_probability)
]

model_comparison = (
    model_comparison
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

display(model_comparison.round(4))

,Model,ROC_AUC,Average_Precision
0,LightGBM Tuned,0.9181,0.5717
1,LightGBM,0.9117,0.5615
2,CatBoost,0.9007,0.5562
3,Logistic Regression,0.8338,0.1840
4,Dummy Baseline,0.5000,0.0344


In [26]:
#  LightGBM (Tuned) - Evaluation Metrics 

threshold_rows = []

for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:

    prediction = (lgbm_tuned_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        prediction,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Predicted_Fraud": prediction.sum(),
        "True_Positive": tp,
        "False_Positive": fp,
        "False_Negative": fn,
        "True_Negative": tn,
        "Precision": precision_score(y_validation, prediction, zero_division=0),
        "Recall": recall_score(y_validation, prediction, zero_division=0),
        "F1_Score": f1_score(y_validation, prediction, zero_division=0),
        "Alert_Rate_Percent": prediction.mean() * 100
    })

threshold_comparison = pd.DataFrame(threshold_rows)

display(threshold_comparison.round(4))

# 0.2 threshold has the best F1 socre [0.5387]

,Threshold,Predicted_Fraud,True_Positive,False_Positive,False_Negative,True_Negative,Precision,Recall,F1_Score,Alert_Rate_Percent
0,0.5,1715,1407,308,2657,113736,0.8204,0.3462,0.4869,1.4521
1,0.4,2061,1586,475,2478,113569,0.7695,0.3903,0.5179,1.7450
2,0.3,2549,1781,768,2283,113276,0.6987,0.4382,0.5386,2.1582
3,0.2,3424,2038,1386,2026,112658,0.5952,0.5015,0.5443,2.8990
4,0.1,5724,2511,3213,1553,110831,0.4387,0.6179,0.5131,4.8464


# Model Comparison Summary

# Data and Validation

- The merged training dataset contains 590,540 transactions.
- A chronological 80/20 train-validation split was used.
- Training transactions: 472,432.
- Validation transactions: 118,108.
- The models were evaluated using the same 433 input features.
- `HasIdentity` and `TranHour` were included as engineered features.

## Models Evaluated
1. Dummy Classifier
2. Logistic Regression
3. CatBoost
4. LightGBM
5. Tuned LightGBM


The Dummy Classifier provided a minimum reference baseline.
Logistic Regression provided a linear baseline but was limited in its ability to capture complex nonlinear fraud patterns.
CatBoost and LightGBM provided substantially stronger performance.

The extended CatBoost model reached its best validation result at iteration [1227].
LightGBM reached its best validation result at iteration [715] and achieved the highest ROC-AUC and Average Precision.

The `tuned LightGBM` model was selected as the final model.
- ROC-AUC improved from 0.9117 to 0.9181.
- Average Precision improved from 0.5615 to 0.5717.
- The best iteration decreased from 715 to 379.


# Selected Business Threshold - `tuned LightGBM` 

A probability threshold of [0.20] was selected as the operational threshold

- Precision: 59.52%
- Recall: 50.15%
- F1-score: 54.43%
- True fraud cases detected: 2,038
- False alerts: 1,386
- Alert rate: 2.90%
This threshold produced the best balance between precision and recall among the evaluated rounded thresholds.

# Feature Importance - LightGBM

The top 20 LightGBM features represented approximately 60.08% of the total model gain.
The strongest features included:

- `V258`
- `C1`
- `DeviceInfo`
- `C14`
- `V294`
- `R_emaildomain`
- `C13`
- `card2`
- `D2`
- `TransactionAmt`

Several important features contain substantial missing values. This supports the earlier finding that missingness can contain predictive information and that features should not be removed based only on their missing percentage.

Feature importance indicates predictive contribution and does not imply a causal relationship with fraud


In [ ]:
from pathlib import Path

output_folder = Path("../../Data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# Calculate feature importance for the tuned model
tuned_feature_importance = pd.DataFrame({
    "Feature": lgbm_tuned_model.booster_.feature_name(),
    "Importance": lgbm_tuned_model.booster_.feature_importance(
        importance_type="gain"
    )
})

tuned_feature_importance["Importance"] = (tuned_feature_importance["Importance"]/ tuned_feature_importance["Importance"].sum()* 100)
tuned_feature_importance = (tuned_feature_importance.sort_values("Importance", ascending=False).reset_index(drop=True))
tuned_feature_importance["Cumulative_Importance"] = (tuned_feature_importance["Importance"].cumsum())

# Select fields required by the two report sections
report_columns = [
    "TransactionID", "TransactionDT", "TranHour",
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "HasIdentity", "DeviceType", "DeviceInfo",
    "P_emaildomain", "R_emaildomain",
    "addr1", "addr2", "dist1", "dist2",
    "D6", "D7", "D12", "D13", "D14",
    "isFraud"
]

# Include the top 20 tuned-model features without duplicates
report_columns = list(dict.fromkeys(report_columns+ tuned_feature_importance.head(20)["Feature"].tolist()))
scored_validation = train_merged.iloc[split_index:][report_columns].copy()

# Verify transaction and prediction alignment
assert scored_validation.index.equals(X_validation_lgbm.index)
assert scored_validation.index.equals(y_validation.index)
assert scored_validation["isFraud"].equals(y_validation)
assert len(scored_validation) == len(lgbm_tuned_probability)
assert scored_validation["TransactionID"].notna().all()
assert scored_validation["TransactionID"].is_unique
assert np.isfinite(lgbm_tuned_probability).all()
assert ((lgbm_tuned_probability >= 0) &
        (lgbm_tuned_probability <= 1)).all()

# Add actual outcomes, predictions, and scoring configuration
scored_validation = scored_validation.rename(columns={"isFraud": "ActualFraud"})

scored_validation["FraudProbability"] = lgbm_tuned_probability
scored_validation["PredictedFraud"] = (
scored_validation["FraudProbability"] >= 0.20).astype("int8")

scored_validation["ModelVersion"] = "LightGBM_Tuned_v1"
scored_validation["Threshold"] = 0.20

# Export the report data and supporting evaluation tables
scored_validation.to_csv(output_folder / "scored_validation_transactions.csv",index=False)
tuned_feature_importance.to_csv(output_folder / "tuned_feature_importance.csv",index=False)
model_comparison.drop_duplicates(subset="Model", keep="last").to_csv(output_folder / "model_comparison.csv",index=False)
threshold_comparison.to_csv(output_folder / "threshold_comparison.csv",index=False)

print("DONE ALhamdullah")

#print(scored_validation.shape)
#print(scored_validation.columns.tolist())

Exported rows and columns: (118108, 41)
Reporting columns:
['TransactionID', 'TransactionDT', 'TranHour', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'HasIdentity', 'DeviceType', 'DeviceInfo', 'P_emaildomain', 'R_emaildomain', 'addr1', 'addr2', 'dist1', 'dist2', 'D6', 'D7', 'D12', 'D13', 'D14', 'ActualFraud', 'V258', 'C1', 'C14', 'V294', 'C13', 'D2', 'id_31', 'D15', 'C11', 'C8', 'C7', 'FraudProbability', 'PredictedFraud', 'ModelVersion', 'Threshold']


# Reporting Data Export

A total of [118,108] validation transactions and [41]were exported for SQL Server analysis and Power BI reporting. 

The chronological validation set was selected because it contains actual fraud labels and predictions from a model trained on the earlier training period.

# Report Scope
* Fraud patterns and important features
* Predictions versus actual outcomes 